## ASSIGNMENT 4

- Date: March 28, 2026
- Course: CSCI E222 Foundations of Large Language Models

In [1]:
import os
from google.colab import files

print("Upload 'News_Summarization_train.csv' and 'News_Summarization_test.csv':")
uploaded = files.upload()

# Verify files exist before proceeding
train_path = "News_Summarization_train.csv"
test_path = "News_Summarization_test.csv"

if os.path.exists(train_path) and os.path.exists(test_path):
    print("✅ Files uploaded successfully.")
else:
    print("❌ Error: Please ensure files are named correctly.")

Upload 'News_Summarization_train.csv' and 'News_Summarization_test.csv':


Saving News_Summarization_test.csv to News_Summarization_test.csv
Saving News_Summarization_train.csv to News_Summarization_train.csv
✅ Files uploaded successfully.


Code for Problem 1: fine-tuned model

In [ ]:
# Install required packages for evaluation
!pip install evaluate rouge_score

In [9]:

import os
import pandas as pd
import numpy as np
import torch
import evaluate
from datasets import Dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    set_seed
)

# Set seed for reproducibility
set_seed(42)

# Setup Device
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Step 1. Load Data
train_df = pd.read_csv("News_Summarization_train.csv")
test_df = pd.read_csv("News_Summarization_test.csv")

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Step 2. Tokenization and Model Loading
model_name = "t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name, legacy=False)
model = T5ForConditionalGeneration.from_pretrained(model_name)

max_input_length = 512
max_target_length = 128

def preprocess_function(examples):
    inputs = ["summarize: " + str(doc) for doc in examples["article"]]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True, padding="max_length")

    labels = tokenizer(text_target=examples["reference_summary"], max_length=max_target_length, truncation=True, padding="max_length")
    # Replace padding token id with -100 to ignore it in loss calculation
    labels_ids = [[(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]]

    model_inputs["labels"] = labels_ids
    return model_inputs

# Map datasets
tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

# Step 3. Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5_base_summarization",
    eval_strategy="no",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=4,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    processing_class=tokenizer,
    data_collator=data_collator,
)

# Step 4. Train
print("Starting fine-tuning...")
trainer.train()
print("Complete fine-tuning model")


Using device: cuda


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Map:   0%|          | 0/2400 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Starting fine-tuning...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Complete fine-tuning model


In [11]:
import numpy as np
import evaluate

# Step 5. Evaluate
prediction_output = trainer.predict(tokenized_test, metric_key_prefix="predict")

predictions = prediction_output.predictions
labels = prediction_output.label_ids

# Process predictions
if isinstance(predictions, tuple):
    predictions = predictions[0]

# 1. Force Argmax if it's 3D (logits)
if predictions.ndim == 3:
    predictions = np.argmax(predictions, axis=-1)

# 2. Filter out extreme values (like -100 or huge garbage numbers)
# Token IDs must be between 0 and the tokenizer's vocab size
vocab_size = tokenizer.vocab_size
predictions = np.where((predictions >= 0) & (predictions < vocab_size), predictions, tokenizer.pad_token_id)

# 3. Explicitly cast to standard python int (via int32)
predictions = predictions.astype(np.int32)
labels = labels.astype(np.int32)

decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

# Process Labels
labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
labels = np.where((labels >= 0) & (labels < vocab_size), labels, tokenizer.pad_token_id)

decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

# Calculate ROUGE scores
rouge = evaluate.load("rouge")
results = rouge.compute(
    predictions=decoded_preds,
    references=decoded_labels,
    use_stemmer=True
)

print(f"\nFinal Evaluation Results:")
print(f"ROUGE-1: {results['rouge1']:.4f}")
print(f"ROUGE-2: {results['rouge2']:.4f}")
print(f"ROUGE-L: {results['rougeL']:.4f}")

# Step 6: Show 3 examples to compare
for i in range(min(3, len(test_df))):
    print(f"\n[Example {i+1}]")
    print(f"Article Excerpt: {test_df['article'].iloc[i][:250].strip()}...")
    print(f"\nReference Summary: {test_df['reference_summary'].iloc[i].strip()}")
    print(f"Generated Summary: {decoded_preds[i].strip()}")
    print("-" * 30)


Final Evaluation Results:
ROUGE-1: 0.4021
ROUGE-2: 0.1916
ROUGE-L: 0.2992

[Example 1]
Article Excerpt: (CNN) -- A Phoenix, Arizona, elementary school bus careened out of control for nearly a mile Wednesday evening, causing more than a dozen accidents and sending at least 26 people to the hospital. A Phoenix, Arizona, school bus crossed over several la...

Reference Summary: Out-of-control school bus crashes into dozens of cars in Phoenix, Arizona .
Panicked children jumped from bus, fleeing into neighborhood .
At least 26 people treated at area hospitals .
Generated Summary: Phoenix, Arizona, school bus careens out of control for nearly a mile . Bus struck two cars as it approached an overpass on Interstate 10 . Bus crossed into oncoming lanes, causing a chain reaction of collisions . At least 26 people were injured in the crash .
------------------------------

[Example 2]
Article Excerpt: (MENTAL FLOSS) -- In the last 2,000 years, commodity shortages, financial speculation, wars, f

### Discussion

The fine-tuned model achieved final evaluation results of **ROUGE-1**: 0.4021, **ROUGE-2**: 0.1916, and **ROUGE-L**: 0.2992, indicating moderate to strong performance on the summarization task. Notably, the ROUGE-1 score reflects good unigram overlap with reference summaries, while the lower ROUGE-2 score suggests that bigram-level fluency and phrase matching remain challenging; the ROUGE-L score shows reasonable sequence-level alignment based on longest common subsequences.

### Hyperparameter Selection

The following hyperparameters were selected to optimize training on the available hardware (e.g., Google Colab T4 GPU):

- **Learning Rate (`5×10⁻5`)** – Allows the pre-trained T5 weights to adapt quickly and effectively to the news dataset.
- **Batch Size (4)** – Fits within GPU VRAM limits (avoiding out-of-memory errors), ensuring stable gradient updates.
- **Training Epochs (4)** – Provides sufficient convergence while preventing overfitting on small/medium datasets.
- **Weight Decay (0.01)** – Applies L2 regularization to improve generalization by penalizing large weights.
- **Mixed Precision (FP16)** – Reduces memory consumption and accelerates computation with minimal accuracy impact.

Code for Problem 2

In [16]:
import torch
import pandas as pd
import numpy as np
import evaluate
from datasets import Dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Clean GPU cache
# Clean GPU cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.backends.cuda.matmul.allow_tf32 = True

device2 = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device2}")

#  Step 1. Load ONLY Test Data
test_df = pd.read_csv("News_Summarization_test.csv")
test_dataset = Dataset.from_pandas(test_df)

# Step 2. Load Vanilla Pre-trained T5
print("Loading original pre-trained T5-base...")
model_name = "t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name, legacy=False)
model = T5ForConditionalGeneration.from_pretrained(model_name).to(device2)

# Step 3. Generation (Inference)
print("Generating summaries (Inference only)...")

def generate_pretrained_summary(batch):
    with torch.no_grad():
        inputs = tokenizer(
            ["summarize: " + str(x) for x in batch["article"]],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(device2)

        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=128,
            num_beams=4,
            early_stopping=True
        )

    return {"pretrained_summary": tokenizer.batch_decode(outputs, skip_special_tokens=True)}

# Run the inference on the test set
test_dataset = test_dataset.map(generate_pretrained_summary, batched=True, batch_size=8)

# Step 4. Compute Metrics
rouge = evaluate.load("rouge")
results = rouge.compute(
    predictions=test_dataset["pretrained_summary"],
    references=test_dataset["reference_summary"],
    use_stemmer=True
)

# Step 5. Final Report
print("\n" + "="*40)
print("  ZERO-SHOT PRE-TRAINED RESULTS")
print("="*40)
print(f"ROUGE-1: {results['rouge1']:.4f}")
print(f"ROUGE-2: {results['rouge2']:.4f}")
print(f"ROUGE-L: {results['rougeL']:.4f}")

# Step 6. Examples
for i in range(min(3, len(test_dataset))):
    print(f"\n--- Example {i+1} ---")
    print(f"Reference: {test_df['reference_summary'].iloc[i].strip()}")
    print(f"Pre-trained: {test_dataset['pretrained_summary'][i].strip()}")

Using device: cuda
Loading original pre-trained T5-base...


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Generating summaries (Inference only)...


Map:   0%|          | 0/600 [00:00<?, ? examples/s]


  ZERO-SHOT PRE-TRAINED RESULTS
ROUGE-1: 0.3641
ROUGE-2: 0.1600
ROUGE-L: 0.2661

--- Example 1 ---
Reference: Out-of-control school bus crashes into dozens of cars in Phoenix, Arizona .
Panicked children jumped from bus, fleeing into neighborhood .
At least 26 people treated at area hospitals .
Pre-trained: bus struck two cars at intersection as it approached an overpass on i-10 . bus crossed into oncoming lanes, causing a chain reaction of collisions . at least two cars overturned; several passengers had to be cut out of wreckage . panicked children began jumping from bus and fled into neighborhood .

--- Example 2 ---
Reference: When in Roman history it was illegal to raise prices, hoard goods or close stores .
Europe's "irrational exuberance" in 1860s was when great buildings were constructed .
Shortages paper and leather led to U.S. bible shortage in 1943 .
Pre-trained: once worthless Roman coins found in the British town of Snodland are considered quite a treasure . when the Roma

## Model Comparison: Fine-tuned vs. Pre-trained

### ROUGE Score Comparison

| Model Version | ROUGE-1 | ROUGE-2 | ROUGE-L |
|---------------|---------|---------|---------|
| Fine-tuned T5-base | 0.4021 | 0.1916 | 0.2992 |
| Pre-trained T5-base | 0.3641 | 0.1600 | 0.2661 |

The fine-tuned T5-base model outperformed the pre-trained baseline across all three ROUGE metrics, achieving gains of +0.038 in ROUGE-1, +0.032 in ROUGE-2, and +0.033 in ROUGE-L. These improvements indicate that fine-tuning successfully enhanced the model's ability to capture unigram, bigram, and sequence-level overlaps with reference summaries.

### Summary Performance

- **Accuracy** – Fine-tuned model better captures key entities and numerical information compared to the pre-trained version.
- **Style & Tone** – Outputs are more fluent and natural, with fewer awkward phrasings.
- **Length** – Summaries are more consistently concise and appropriately scaled.
- **Coherence** – Improved sentence flow and logical progression of ideas.

### Discussion

Quantitatively, fine-tuning led to consistent improvements across all ROUGE metrics, with the most notable gain in ROUGE-1 (unigram overlap). Qualitatively, the fine-tuned model produces more accurate, coherent, and naturally phrased summaries than the pre-trained model.

**Conclusion:** Fine-tuning T5-base on the news summarization dataset was effective, yielding meaningful performance gains over the pre-trained model.